# Phase 6: Pricing Analytics

## Objective

Assess observed unit-price patterns and listed-price reference differences without claiming discounts, elasticity, causality, or profit margin.

**Scope control:** Amazon selling-price proxies and March/May platform MRP snapshots are analysed separately. No Amazon-to-product SKU match exists, so no realised discount calculation is performed.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
try:
    from IPython.display import display
except ImportError:
    def display(*objects):
        for obj in objects: print(obj)
ROOT = Path.cwd()
if not (ROOT / 'data').exists(): ROOT = ROOT.parent
from src.status_scope import add_status_scope
amazon = pd.read_csv(ROOT / 'data/cleaned/amazon_sale_report_cleaned.csv', low_memory=False)
may = pd.read_csv(ROOT / 'data/cleaned/may_2022_cleaned.csv', low_memory=False)
march = pd.read_csv(ROOT / 'data/cleaned/p_l_march_2021_cleaned.csv', low_memory=False)
amazon['date'] = pd.to_datetime(amazon['date'], errors='coerce')
for col in ['amount', 'qty']: amazon[col] = pd.to_numeric(amazon[col], errors='coerce')
amazon = add_status_scope(amazon)
delivered = amazon[amazon['is_delivered_status_proxy']].copy()
print('Loaded Amazon:', amazon.shape, '| delivered proxy:', delivered.shape, '| May:', may.shape, '| March:', march.shape)


## 1. Definitions, units, and currency

**Observation:** Amazon has INR on populated currency rows; international `rate` and `gross_amt` have no currency field.

**Evidence:** Amazon prices are calculated only from `currency == 'INR'`, non-null `amount`, and `qty > 0`. Product snapshot MRP fields are reference values with no confirmed accounting definition.

**Interpretation:** Amazon unit price is a line-level reported amount per unit. Snapshot values are listed/reference prices, not realised selling prices.

**Business implication:** Keep the two price populations separate and label all outputs accordingly.

**Limitation:** No FX conversion or cross-source price comparison is possible.

In [ ]:
assert amazon['currency'].dropna().eq('INR').all()
assert delivered['currency'].dropna().eq('INR').all()
print('Amazon currency values:', amazon['currency'].value_counts(dropna=False).to_dict())
print('International currency: unavailable; international rate is excluded from cross-source price analysis.')
print('Amazon amount coverage:', f'{amazon["amount"].notna().mean():.1%}', '| delivered proxy:', f'{delivered["amount"].notna().mean():.1%}')


## 2. Amazon observed unit-price distribution

**Observation:** A selling-price proxy can be calculated for delivered-status-proxy rows with positive quantity and populated INR amount.

**Evidence:** `unit_price = amount / qty` at Amazon line grain. Zero and negative prices are checked and retained as anomaly records.

**Interpretation:** The distribution describes observed reported amount per unit within one status proxy.

**Business implication:** Use the distribution to identify price-quality questions and price bands for descriptive monitoring.

**Limitation:** This is not a net realised price or elasticity estimate.

In [ ]:
price_rows = delivered[delivered['currency'].eq('INR') & delivered['amount'].notna() & delivered['qty'].gt(0)].copy()
price_rows['unit_price'] = price_rows['amount'] / price_rows['qty']
price_summary = price_rows['unit_price'].describe(percentiles=[.01, .25, .5, .75, .95, .99]).to_frame('unit_price_inr')
display(price_summary)
print('Zero unit-price rows:', int((price_rows['unit_price'] == 0).sum()), '| negative:', int((price_rows['unit_price'] < 0).sum()))
fig, ax = plt.subplots(figsize=(8, 4))
price_rows['unit_price'].plot.hist(bins=40, ax=ax, color='#486581')
ax.set_title('Amazon delivered-status-proxy unit-price distribution', loc='left', fontweight='bold')
ax.set_xlabel('Reported amount per unit (INR)')
ax.set_ylabel('Line count')
plt.tight_layout(); plt.show()


## 3. Listed/reference prices and platform differences

**Observation:** March and May snapshots contain comparable platform-labelled MRP fields, but they are not linked to Amazon sales.

**Evidence:** The snapshot-level analysis compares non-null fields within the same SKU row and reports ranges and pairwise differences.

**Interpretation:** These are reference-price inconsistencies within the snapshots, not marketplace selling-price differences.

**Business implication:** Investigate catalogue governance where platform reference prices diverge.

**Limitation:** Platform field definitions, timing, and commercial comparability are not confirmed; MRP is not cost.

In [ ]:
platform_cols = ['ajio_mrp', 'amazon_mrp', 'amazon_fba_mrp', 'flipkart_mrp', 'limeroad_mrp', 'myntra_mrp', 'paytm_mrp', 'snapdeal_mrp']
for frame in [may, march]:
    for col in platform_cols: frame[col] = pd.to_numeric(frame[col], errors='coerce')
snapshot = may[['sku'] + platform_cols].copy()
platform_summary = snapshot[platform_cols].agg(['count', 'mean', 'median', 'min', 'max']).T.sort_values('mean', ascending=False)
snapshot['reference_price_range'] = snapshot[platform_cols].max(axis=1) - snapshot[platform_cols].min(axis=1)
snapshot['platform_price_count'] = snapshot[platform_cols].notna().sum(axis=1)
display(platform_summary)
display(snapshot.sort_values('reference_price_range', ascending=False).head(10))
print('Rows with at least two platform prices:', int(snapshot['platform_price_count'].ge(2).sum()))
print('Rows with non-zero platform-price range:', int(snapshot['reference_price_range'].gt(0).sum()))


## 4. Discount support and listed price versus selling price

**Observation:** Actual discount amount and discount percentage are not feasible from the current data.

**Evidence:** Amazon SKUs have zero exact matches to the March/May product snapshots, and `amount` is line-level while snapshot MRP fields are separate reference rows.

**Interpretation:** A subtraction such as `MRP - amount` would mix unmatched products, different grains, and unconfirmed price definitions.

**Business implication:** Do not report discount rate, high-discount products, or realised price-versus-MRP comparisons until a governed SKU crosswalk and comparable time basis exist.

**Limitation:** `promotion_ids` indicates promotion presence only; it does not contain discount value.

In [ ]:
matched_rows = int(amazon['sku'].isin(may['sku']).sum())
discount_feasibility = pd.DataFrame({'metric': ['Discount amount', 'Discount percentage', 'Selling price versus MRP'], 'status': ['Not feasible', 'Not feasible', 'Not feasible'], 'reason': ['No discount amount and no valid SKU-MRP join', 'No discount amount and no valid SKU-MRP join', 'Amazon SKU format has zero exact matches to product snapshot SKUs']})
display(discount_feasibility)
print('Exact Amazon-to-May SKU matches:', matched_rows)
assert matched_rows == 0


## 5. Price variation by SKU and category

**Observation:** Within-source price variation can be described for delivered-status-proxy Amazon rows with valid unit prices.

**Evidence:** SKU and category summaries use median, mean, minimum, maximum, line count, and units from the same valid-price population.

**Interpretation:** Differences describe observed mix and variation, not price elasticity or optimal pricing.

**Business implication:** Use high-variation groups for catalogue and pricing-data review.

**Limitation:** Price variation may reflect size, product mix, status, promotions, or data quality.

In [ ]:
sku_price = price_rows.groupby('sku').agg(median_unit_price=('unit_price','median'), mean_unit_price=('unit_price','mean'), min_unit_price=('unit_price','min'), max_unit_price=('unit_price','max'), lines=('unit_price','size'), units=('qty','sum')).sort_values('lines', ascending=False)
sku_price['price_range'] = sku_price['max_unit_price'] - sku_price['min_unit_price']
category_price = price_rows.groupby('category', dropna=False).agg(median_unit_price=('unit_price','median'), mean_unit_price=('unit_price','mean'), min_unit_price=('unit_price','min'), max_unit_price=('unit_price','max'), lines=('unit_price','size'), units=('qty','sum')).sort_values('lines', ascending=False)
category_price['price_range'] = category_price['max_unit_price'] - category_price['min_unit_price']
display(sku_price.head(10), category_price)


## 6. Sales volume by price band

**Observation:** Descriptive volume bands are mutually exclusive and cover all non-negative valid unit prices.

**Evidence:** Bands are `[0,500)`, `[500,1000)`, `[1000,2000)`, and `[2000,inf)` INR per unit using left-closed, right-open intervals.

**Interpretation:** The result describes where observed units and amount fall; it does not estimate demand response to price.

**Business implication:** Use bands for monitoring and sample review, not price recommendations.

**Limitation:** Boundaries are analytical groupings, not approved commercial thresholds.

In [ ]:
band_edges = [0, 500, 1000, 2000, np.inf]
band_labels = ['[0,500)', '[500,1000)', '[1000,2000)', '[2000,inf)']
price_rows['price_band'] = pd.cut(price_rows['unit_price'], bins=band_edges, labels=band_labels, right=False, include_lowest=True)
band_summary = price_rows.groupby('price_band', observed=False).agg(lines=('unit_price','size'), units=('qty','sum'), reported_amount=('amount','sum'))
display(band_summary)
assert band_summary['lines'].sum() == len(price_rows)
assert price_rows['price_band'].notna().all()
assert len(set(band_labels)) == len(band_labels)


## 7. Price anomalies and validation

**Observation:** Zero unit prices are present because some valid-quantity lines have zero reported amount; extreme values are flagged using an IQR rule and retained.

**Evidence:** The anomaly table shows zero/negative checks and high-price candidates. Manual arithmetic is reconciled on a sample of valid lines.

**Interpretation:** Anomaly flags require source review and are not automatic price corrections.

**Business implication:** Investigate zero-amount lines and high unit-price lines with the source owner before using them in pricing decisions.

**Limitation:** No business-approved anomaly threshold or price correction rule exists.

In [ ]:
zero_price_rows = delivered[delivered['qty'].gt(0) & delivered['amount'].eq(0)]
negative_price_rows = delivered[delivered['qty'].gt(0) & delivered['amount'].lt(0)]
q1, q3 = price_rows['unit_price'].quantile([0.25, 0.75])
upper_price_bound = q3 + 1.5 * (q3 - q1)
high_price_rows = price_rows[price_rows['unit_price'] > upper_price_bound].sort_values('unit_price', ascending=False)
sample = price_rows[['amount', 'qty', 'unit_price']].head(10).copy()
sample['manual_unit_price'] = sample['amount'] / sample['qty']
display(pd.DataFrame({'check': ['zero unit price', 'negative unit price', 'high IQR unit price', 'invalid qty denominator'], 'count': [len(zero_price_rows), len(negative_price_rows), len(high_price_rows), int((price_rows['qty'] <= 0).sum())]}))
display(sample)
assert np.allclose(sample['unit_price'], sample['manual_unit_price'])
assert (price_rows['qty'] > 0).all()
print('IQR upper bound:', f'{upper_price_bound:,.2f}', '| high-price rows:', len(high_price_rows))
print('No anomaly rows were removed.')


## 8. Final exclusions and limitations

- Discount amount, discount percentage, high-discount products, and realised price-versus-MRP are not calculated.
- No price elasticity, causal claim, or specific price increase is recommended.
- MRP and TP fields are not treated as cost.
- Amazon and international price fields are not combined because currency is unavailable for international data.
- The delivered status is a sensitivity proxy, not a confirmed completed-sales rule.
- The March/May product snapshots remain separate because the Amazon SKU crosswalk is missing.